# OpenRouter Smoke Test

Run this notebook top to bottom to check whether OpenRouter is failing outside the full digest pipeline.

It does three things:
1. loads the OpenRouter API key from the usual env file
2. makes a tiny JSON smoke call against `qwen/qwen3-coder:free`
3. makes the actual February 2026 ZUNA triage call and shows the raw response shape plus the repo wrapper output


In [ ]:
from pathlib import Path
import json
import os
import traceback

from dotenv import load_dotenv
from openai import OpenAI

from eegfm_digest.llm import LLMCallConfig, OpenAICall

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
ENV_PATH = Path("~/2_cs_projects/env/llm_api/.env").expanduser()
MODEL = "qwen/qwen3-coder:free"
BASE_URL = "https://openrouter.ai/api/v1"

if ENV_PATH.exists():
    load_dotenv(ENV_PATH, override=True)

api_key = os.environ.get("OPENROUTER_API_KEY")
if not api_key:
    raise RuntimeError(f"Missing OPENROUTER_API_KEY. Checked env file at {ENV_PATH}")

print(f"repo_root={REPO_ROOT}")
print(f"env_path={ENV_PATH}")
print(f"model={MODEL}")


: 

In [ ]:
client = OpenAI(api_key=api_key, base_url=BASE_URL)

def preview_response(resp):
    print(type(resp))
    choice = resp.choices[0] if getattr(resp, "choices", None) else None
    if choice is None:
        print("no choices on response")
        return
    message = getattr(choice, "message", None)
    print("message.content:")
    print(getattr(message, "content", None))
    print("finish_reason:", getattr(choice, "finish_reason", None))
    if hasattr(resp, "model_dump"):
        dumped = resp.model_dump()
        print("top-level keys:", sorted(dumped.keys()))
        print("first choice keys:", sorted(dumped.get("choices", [{}])[0].keys()))

wrapper = OpenAICall(
    LLMCallConfig(
        provider="openrouter",
        api_key=api_key,
        model=MODEL,
        temperature=0.2,
        max_output_tokens=1024,
        base_url=BASE_URL,
    )
)


In [ ]:
smoke_prompt = 'Return exactly this JSON object and nothing else: {"ok": true, "provider": "openrouter", "model": "qwen3-coder"}'

try:
    smoke_resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": smoke_prompt}],
        temperature=0.2,
        max_tokens=256,
        response_format={"type": "json_object"},
    )
    print("DIRECT SMOKE CALL")
    preview_response(smoke_resp)
except Exception:
    print("DIRECT SMOKE CALL FAILED")
    traceback.print_exc()

try:
    wrapper_result = wrapper.call(smoke_prompt, schema={"type": "object"})
    print("WRAPPER SMOKE CALL")
    print(wrapper_result.text)
except Exception:
    print("WRAPPER SMOKE CALL FAILED")
    traceback.print_exc()


In [ ]:
raw_candidates = json.loads((REPO_ROOT / "outputs" / "2026-02" / "arxiv_raw.json").read_text(encoding="utf-8"))
paper = next(item for item in raw_candidates if item["arxiv_id_base"] == "2602.18478")
triage_template = (REPO_ROOT / "prompts" / "triage.md").read_text(encoding="utf-8")
zuna_prompt = triage_template.replace("{{TITLE}}", paper["title"]).replace("{{ABSTRACT}}", paper["summary"])

print(paper["title"])
print()
print(zuna_prompt[:800])
print("...")


In [ ]:
try:
    zuna_resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": zuna_prompt}],
        temperature=0.2,
        max_tokens=1024,
        response_format={"type": "json_object"},
    )
    print("DIRECT ZUNA TRIAGE CALL")
    preview_response(zuna_resp)
except Exception:
    print("DIRECT ZUNA TRIAGE CALL FAILED")
    traceback.print_exc()

try:
    wrapper_result = wrapper.call(zuna_prompt, schema={"type": "object"})
    print("WRAPPER ZUNA TRIAGE CALL")
    print(wrapper_result.text)
except Exception:
    print("WRAPPER ZUNA TRIAGE CALL FAILED")
    traceback.print_exc()
finally:
    wrapper.close()
